# Interactive Notebook 05 - PMSM operation point selection:

This interactive Jupyter notebook introduces the concept of operating point selection for permanent magnet synchronous motors.

For help with the installation of the required software, consider the comments in ```CTPD_course\interactive_notebooks\README.md```.
Throughout the exercises, we will be using a combination of scientific computation libraries from the [JAX](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html) ecosystem and visualize them with [matplotlib](https://matplotlib.org/) and [ipywidgets](https://ipywidgets.readthedocs.io/en/stable/).

### Preliminaries & Imports:

In [ ]:
# automatically reloads imported ```.py```-files once they are changed and saved
%load_ext autoreload
%autoreload 2

In [ ]:
%%html
<style>
div.jupyter-widgets.widget-label {display: none;}
</style>

In [ ]:
# imports required packages
from functools import partial
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.io import loadmat
from pathlib import Path
from matplotlib import rc
mpl.rcParams.update({'font.size': 20})

import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

(**Optional**: If you have LaTeX installed, you can use the following lines for pretty rendering of plot labels.
Any LaTeX installation should work, as long as all the required packages are installed, e.g., [MiKTeX](https://miktex.org/) or [TeXLive](https://www.tug.org/texlive/).

If you do not have LaTeX installed, you can comment the next cell out or skip it.)

In [ ]:
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

---

**Throught the whole notebook, we will assume that the rotation speed $\omega$ can be set arbitrarily from an external load machine.**

### Operating point selection for the nonlinear PMSM:

We will use an adapted implementation from inb03 as our plant to be controlled (code in `utils/pmsm.py`) and the current controller from inb04 (code in `utils/current_controller.py`).

Now we want to control the system to produce a specific torque $T$.

We are looking at the operating point selection (OPS) component of our complete control structure:

<div>
<img src="fig/OPS_structure_diagramm.png" width="1000"/>
</div>

In [ ]:
# starting with this notebook, all auxiliary simulation code for the PMSM will be placed in 
# `CTPD/interactive_notebooks/utils/`
import sys

from path_helper import get_folder_path
sys.path.insert(1, str(get_folder_path()))

In [ ]:
from utils.pmsm import PMSM, lut_interpolate, clip_voltage
from utils.motor_params import MotorVariant
from utils.visualization import visualize_trajectories, visualize_trajectories_with_reference
from utils.signals import aprbs
from utils.current_controller import FOCController

Below we initialize the PMSM simulator and the current controller that you already know.

In [ ]:
pmsm = PMSM(
    saturated=True,
    motor_variant=MotorVariant.BRUSA,
    T_s=1e-4,
)
pmsm

In [ ]:
current_controller = FOCController(
    static_params=pmsm.env_properties.static_params,
    lut_grids=pmsm.LUT_grids,
    lut_values=pmsm.LUT_values,
    T_s=pmsm.T_s,
    d=0.99,
    rho=0.25,
) 
current_controller

Next we want to build a structure that computes target currents based on a target torque.
The plan is to build a real-time map $T^* \rightarrow i^*$ in the form of a look-up-table (LUT).

The structure of the LUT-based OPS is:

<div>
<img src="fig/LUT-based_OPS.jpg" width="1000"/>
</div>

We need to compute 3 LUTs to be able to use this.

The `flux_MTPC_LUT`:
$$ \large \psi^*_{\mathrm{MTPC}} (T^*) = \| \boldsymbol{\psi}_{\mathrm{s, dq}} (i^*_{\mathrm{MTPC}} (T^*))\|$$
which yields the required flux magnitude for the requsted torque.

The `torque_MTPV_LUT`:
$$ \large T_{\mathrm{lim}} (\psi_{\mathrm{lim}}) = \max_{\boldsymbol{i}} T(\boldsymbol{i})$$
which yields maximum achievable torque for a given $\psi_{\mathrm{lim}} \approx u_{\mathrm{max}} / \omega$.

The `flux_torque_to_current_LUT`:
$$ \large \begin{bmatrix} i^*_{\mathrm{s,d}} & i^*_{\mathrm{s,q}} \end{bmatrix} = f(\psi^*_{\mathrm{lim}}, T^*_{\mathrm{lim}})$$
which yields the current operating point in the feasible operating region.

$\psi^*_{\mathrm{lim}}, T^*_{\mathrm{lim}}$ are the clipped